# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
29,0.71,710.5,269.5,220.5,3.5,0.0,9.060
123,0.74,686.0,245.0,220.5,3.5,0.1,12.120
186,0.64,784.0,343.0,220.5,3.5,0.1,17.245
559,0.71,710.5,269.5,220.5,3.5,0.4,16.025
663,0.66,759.5,318.5,220.5,3.5,0.4,16.560


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

> CEVABINIZI BURAYA YAZIN
> mse
> 

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [7]:
from sklearn.preprocessing import StandardScaler
X=data.drop(columns="Average Temperature")
y=data["Average Temperature"]
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [27]:
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDRegressor

from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate

sgd_model = SGDRegressor(loss="squared_error", random_state=42)

cv_results = cross_validate(
    sgd_model,
    X, y,
  
    cv=10,
    scoring=[ "r2", "max_error"]
)


cv_results

{'fit_time': array([0.00709105, 0.00592589, 0.00565004, 0.0042491 , 0.003196  ,
        0.00317168, 0.00342298, 0.00343108, 0.00418782, 0.00367904]),
 'score_time': array([0.00262713, 0.001894  , 0.00167012, 0.00136709, 0.00137377,
        0.0012722 , 0.00102115, 0.0008738 , 0.00084305, 0.00082493]),
 'test_r2': array([-6.17456732e+26, -8.92790500e+26, -6.30071744e+26, -5.42648955e+26,
        -1.49252290e+27, -1.02750745e+27, -6.91614494e+25, -2.60576053e+25,
        -7.29094801e+26, -2.35215170e+26]),
 'test_max_error': array([-2.86283353e+14, -3.02912873e+14, -2.78904024e+14, -2.66116927e+14,
        -4.17245058e+14, -3.94634953e+14, -9.85733288e+13, -8.65510021e+13,
        -3.19730277e+14, -1.73647962e+14])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [22]:
r2=cv_results["test_r2"].mean()
r2
max_error_celsius= cv_results["test_max_error"].max()
r2, max_error_celsius

(-6.262527311354223e+26, -86551002118018.72)

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [24]:
mae_model = SGDRegressor(loss="epsilon_insensitive", epsilon = 0)
mae_cv=cross_validate(mae_model,
                      X,y,
                      cv=10,
                      scoring= ['r2','max_error'])


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [26]:
r2_mae=mae_cv["test_r2"].mean()
max_error_mae=mae_cv["test_max_error"].max()
r2_mae, max_error_mae

(-233.72388022242907, -65.6013772211834)

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> CEVABINIZI BURAYA YAZIN
> 2. mae 

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())